# 📘 Tutorial 6: Wide, Long, and Tidy Data

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

Plotting multiple series is much easier when the table shape matches the question. This tutorial introduces wide data, long data, and tidy data using `melt` and `pivot`.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- distinguish wide and long data,
- use `melt` to create long-form data,
- use `pivot` or `pivot_table` to return to wide form,
- explain why long-form data is often easier for grouped plotting,
- avoid confusing index and column states.


## 🔧 Setup


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Read a wide table


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

wide_path = project_root / "data" / "part1" / "wide_signals.csv"
wide = pd.read_csv(wide_path)
wide


This table is wide because each sample is stored in a separate column.


## 2. Why wide data can be awkward


In [ ]:
response_columns = ["sample_A", "sample_B", "sample_C"]
wide[response_columns].mean()


This works for simple summaries, but the sample identifier is trapped inside column names. If we want to group by sample metadata later, long data is usually cleaner.


## 3. Convert wide data to long data with `melt`


In [ ]:
long = wide.melt(
    id_vars="time_min",
    var_name="sample_id",
    value_name="response",
)

long


In [ ]:
long["sample_id"] = long["sample_id"].str.replace("sample_", "", regex=False)
long


In long form, each row is one observation. The sample identifier is now a value in a column rather than part of a column name.


## 4. Add metadata to the long table


In [ ]:
sample_metadata = pd.DataFrame(
    {
        "sample_id": ["A", "B", "C"],
        "condition": ["control", "control", "treated"],
    }
)

tidy = long.merge(sample_metadata, on="sample_id", how="left")
tidy


This is tidy data: each variable has its own column, each observation has its own row, and each value has its own cell.


## 5. Why long-form data helps plotting


In [ ]:
tidy[["time_min", "response", "condition", "sample_id"]].head()


A later plotting tutorial could use:

- `time_min` as x,
- `response` as y,
- `sample_id` or `condition` as a grouping variable.


## 6. Return to wide form with `pivot`


In [ ]:
wide_again = tidy.pivot(
    index="time_min",
    columns="sample_id",
    values="response",
).reset_index()

wide_again.columns.name = None
wide_again


`pivot` uses unique combinations of index and column values. After pivoting, `reset_index` turns `time_min` back into a normal column.


## 7. Use `pivot_table` when duplicates exist


In [ ]:
duplicated = pd.concat([tidy, tidy.iloc[[0]]], ignore_index=True)

duplicated.head()


In [ ]:
duplicated.pivot_table(
    index="time_min",
    columns="sample_id",
    values="response",
    aggfunc="mean",
).reset_index()


`pivot_table` can aggregate duplicate combinations. That is useful, but it should be a deliberate choice.


## 8. Checkpoint: create a treated-only long table

Task: select only rows where `condition` is treated and keep the columns needed for a later grouped plot.


In [ ]:
treated_long = tidy.loc[
    tidy["condition"] == "treated",
    ["time_min", "sample_id", "condition", "response"],
].reset_index(drop=True)

treated_long


## 🧭 Key takeaways

- Wide data stores multiple series in separate columns.
- Long data stores one observation per row.
- `melt` is the main tool for going from wide to long.
- `pivot` and `pivot_table` are useful for returning to wide summaries.
- Reset indexes deliberately so plotting columns remain ordinary columns.
